# Flash Attention 2 Handwrite Backward

手撕Flash Attention 2

Author： xiaodongguaAIGC

1. 实现attention前向计算
2. 实现attention autograd 梯度计算
3. 实现attention 手写梯度计算
4. 实现Flash Attention 2 除法版本版本
5. 实现Flash Attention 2 非除法计算版本
6. 实现Flash Attention 2 手写backward
7. 分析Flash Attention 2 梯度计算量

In [147]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
torch.manual_seed(42)

In [148]:
dim = 3 # d_model
n = 6 # seq_length
block = 2 # block
nb = n // block # seq per block

In [149]:
X_src = torch.randn(n, dim)
Y_label = torch.randn(n, dim)

# Forward

In [150]:
class attention(nn.Module):
    def __init__(self, dim):
        super(attention, self).__init__()
        self.dim = dim
        self.w = nn.Linear(dim, dim, bias = False)
        self.wq = nn.Linear(dim, dim, bias = False)
        self.wk = nn.Linear(dim, dim, bias = False)
        self.wv = nn.Linear(dim, dim, bias = False)
        self.wo = nn.Linear(dim, dim, bias = False)
        
    def forward(self, X_src):
        X = self.w(X_src)
        Q, K, V = self.wq(X), self.wk(X), self.wv(X)
        S = Q @ K.t() / math.sqrt(dim)
        S_softmax = F.softmax(S, dim = 1)
        O = S_softmax @ V
        Y = self.wo(O)
        return Y, X, Q, K, V, S, S_softmax, O

model = attention(dim)
Y, X, Q, K, V, S, S_softmax, O = model(X_src)
X.retain_grad()
O.retain_grad()
Y.retain_grad()
V.retain_grad()
S_softmax.retain_grad() 
S.retain_grad()
Q.retain_grad()
K.retain_grad()
X.retain_grad()

# 其中 # 可以做成Gradient checkpoint

# Backward AutoGrad

In [151]:
loss = (0.5 * (Y - Y_label) ** 2).mean() # MSE Loss
loss.backward()
print('grad wo:', model.wo.weight.grad)
print('grad wq:', model.wq.weight.grad)
print('grad wk:', model.wk.weight.grad)
print('grad wv:', model.wv.weight.grad)
print('grad w:', model.w.weight.grad)

grad wo: tensor([[-4.6842e-03, -4.3981e-03, -6.1333e-03],
        [-4.7800e-02, -8.8314e-03, -2.2341e-02],
        [-6.9370e-03, -7.7704e-05, -1.6573e-03]])

grad wq: tensor([[ 0.0119, -0.0078, -0.0032],
        [-0.0017,  0.0010, -0.0001],
        [-0.0093,  0.0058,  0.0005]])

grad wk: tensor([[ 7.5153e-05, -1.0956e-04,  1.6268e-03],
        [-2.2055e-03,  2.3473e-03, -1.1345e-03],
        [-9.4649e-03,  9.8067e-03, -8.4446e-03]])

grad wv: tensor([[-0.0081,  0.0093, -0.0190],
        [-0.0189,  0.0210, -0.0426],
        [-0.0135,  0.0151, -0.0334]])

grad w: tensor([[-0.0116, -0.0019,  0.0031],
        [-0.0096, -0.0086, -0.0060],
        [-0.0391, -0.0268,  0.0029]])

In [152]:
print('grad X:', X.grad)
print('grad O:', O.grad)
print('grad Y:', Y.grad)
print('grad V:', V.grad)
print('grad S_softmax:', S_softmax.grad)
print('grad S:', S.grad)
print('grad Q:', Q.grad)
print('grad K:', K.grad)

grad X: tensor([[-6.7394e-03, -2.2583e-03, -1.6907e-02],
        [-4.1190e-03, -3.5749e-03, -6.9841e-03],
        [ 2.7661e-03, -4.1038e-03, -4.1821e-03],
        [ 1.0400e-03, -3.9575e-03, -1.2644e-03],
        [-7.1551e-03, -5.7066e-03, -1.3210e-02],
        [ 7.3263e-05, -5.1323e-03, -4.6628e-03]])

grad O: tensor([[ 0.0635,  0.0602,  0.0476],
        [ 0.0189,  0.0181,  0.0334],
        [-0.0584, -0.0424, -0.0468],
        [-0.0503,  0.0001, -0.0055],
        [ 0.0746,  0.0311,  0.0187],
        [-0.0245, -0.0064,  0.0033]])

grad Y: tensor([[ 0.0046, -0.1124,  0.0480],
        [-0.0738,  0.0330,  0.0527],
        [ 0.0388,  0.0357, -0.0825],
        [-0.0166, -0.0069, -0.0906],
        [ 0.0576, -0.0895,  0.0787],
        [-0.0473,  0.0462, -0.0159]])

grad V: tensor([[0.0059, 0.0131, 0.0100],
        [0.0035, 0.0085, 0.0076],
        [0.0052, 0.0115, 0.0090],
        [0.0038, 0.0088, 0.0077],
        [0.0026, 0.0093, 0.0082],
        [0.0028, 0.0095, 0.0082]])

grad S_softmax: tensor([[ 0.1132,  0.0035,  0.0842,  0.0045, -0.0098,  0.0196],
        [ 0.0403,  0.0069,  0.0328,  0.0072, -0.0050,  0.0061],
        [-0.1050,  0.0017, -0.0777, -0.0021,  0.0079, -0.0175],
        [-0.0778,  0.0282, -0.0503,  0.0170, -0.0010, -0.0130],
        [ 0.1195, -0.0247,  0.0811, -0.0150, -0.0031,  0.0208],
        [-0.0359,  0.0126, -0.0225,  0.0086, -0.0004, -0.0065]])

grad S: tensor([[ 0.0168, -0.0048,  0.0071, -0.0056, -0.0087, -0.0048],
        [ 0.0042, -0.0012,  0.0029, -0.0011, -0.0035, -0.0014],
        [-0.0150,  0.0051, -0.0071,  0.0052,  0.0077,  0.0040],
        [-0.0100,  0.0072, -0.0055,  0.0052,  0.0026,  0.0004],
        [ 0.0146, -0.0093,  0.0087, -0.0075, -0.0052, -0.0013],
        [-0.0051,  0.0032, -0.0025,  0.0027,  0.0013,  0.0003]])

grad Q: tensor([[-0.0100,  0.0019,  0.0094],
        [-0.0024,  0.0007,  0.0029],
        [ 0.0094, -0.0017, -0.0086],
        [ 0.0089, -0.0009, -0.0057],
        [-0.0123,  0.0015,  0.0086],
        [ 0.0043, -0.0004, -0.0027]])

grad K: tensor([[-8.4886e-04,  4.4063e-04,  4.6638e-03],
        [-8.1445e-04, -9.8046e-04, -1.5524e-03],
        [ 8.5437e-05,  6.6892e-04,  1.5607e-03],
        [-2.1524e-04, -4.8444e-04, -1.7435e-03],
        [ 9.8139e-04,  5.9550e-05, -1.8047e-03],
        [ 8.1172e-04,  2.9579e-04, -1.1239e-03]])

### Backward Hand-write

In [104]:
N = n * dim 

In [105]:
dY =  Y - Y_label
dY = 1 / N * dY
print(dY)

tensor([[ 0.0125, -0.0958,  0.0261],
        [-0.0661,  0.0450,  0.0403],
        [ 0.0466,  0.0508, -0.1017],
        [-0.0089,  0.0050, -0.1035],
        [ 0.0659, -0.0773,  0.0657],
        [-0.0393,  0.0594, -0.0306]], grad_fn=<MulBackward0>)

In [106]:
dwo = O.t() @ dY 
print(dwo)

tensor([[ 0.0003, -0.0003, -0.0022],
        [-0.0004,  0.0003,  0.0021],
        [-0.0016,  0.0013,  0.0029]], grad_fn=<MmBackward0>)

In [107]:
# Y = O @ Wo     n x d = n x d @ d x d 
dO = dY @ model.wo.weight
print(dO)

tensor([[-0.0111, -0.0205, -0.0027],
        [-0.0189,  0.0062,  0.0263],
        [ 0.0489,  0.0227, -0.0408],
        [ 0.0546,  0.0239, -0.0496],
        [-0.0356, -0.0332,  0.0200],
        [ 0.0163,  0.0198, -0.0056]], grad_fn=<MmBackward0>)

In [108]:
# O = S_softmax @ V    n x d = n x n @ n x d
dV = S_softmax.t() @ dO
print(dV)

tensor([[ 0.0089,  0.0033, -0.0085],
        [ 0.0094,  0.0032, -0.0091],
        [ 0.0089,  0.0031, -0.0086],
        [ 0.0090,  0.0029, -0.0089],
        [ 0.0089,  0.0031, -0.0086],
        [ 0.0091,  0.0033, -0.0087]], grad_fn=<MmBackward0>)

In [109]:
# V = Wv @ X    # n x d = n x d @ d x d
dwv = dV.t() @ X
print(dwv)

tensor([[ 0.0056, -0.0023,  0.0007],
        [ 0.0018, -0.0005,  0.0005],
        [-0.0055,  0.0024, -0.0005]], grad_fn=<MmBackward0>)

In [110]:
# O = S_softmax @ V
dS_softmax = dO @ V.t()
print(dS_softmax)

tensor([[-0.0020,  0.0025, -0.0003,  0.0024, -0.0005, -0.0007],
        [ 0.0145, -0.0148,  0.0017, -0.0168,  0.0028,  0.0060],
        [-0.0215,  0.0211, -0.0023,  0.0250, -0.0040, -0.0093],
        [-0.0262,  0.0254, -0.0028,  0.0304, -0.0048, -0.0114],
        [ 0.0099, -0.0087,  0.0008, -0.0115,  0.0016,  0.0046],
        [-0.0025,  0.0018, -0.0001,  0.0029, -0.0003, -0.0013]],
       grad_fn=<MmBackward0>)

In [133]:
# S_softmax = softmax(S)
# gradient = diag(S_softmax) - S_softmax X S_softmax

dS = torch.zeros_like(S_softmax)
for i in range(n):
    I = torch.diag(S_softmax[i,:]) - torch.outer(S_softmax[i,:], S_softmax[i,:])
    dS[i,:] = dS_softmax[i,:] @ I
print(dS)

tensor([[-3.7453e-04,  3.7865e-04, -9.5388e-05,  3.6472e-04, -1.2289e-04,
         -1.5057e-04],
        [ 2.6533e-03, -2.2899e-03,  4.1045e-04, -2.5316e-03,  5.9628e-04,
          1.1615e-03],
        [-3.8123e-03,  3.2790e-03, -6.6126e-04,  3.9444e-03, -9.3226e-04,
         -1.8176e-03],
        [-4.6737e-03,  4.0552e-03, -7.4735e-04,  4.6846e-03, -1.0796e-03,
         -2.2392e-03],
        [ 1.7250e-03, -1.3623e-03,  2.4026e-04, -1.8352e-03,  3.6702e-04,
          8.6524e-04],
        [-4.2494e-04,  2.8079e-04, -3.1284e-05,  4.6324e-04, -6.3356e-05,
         -2.2444e-04]], grad_fn=<CopySlices>)

In [136]:
# S = Q @ K.t() / sqrt(dim)
dQ = dS @ K  / math.sqrt(dim)
print(dQ)

tensor([[-3.7311e-04, -3.4259e-04,  4.6313e-05],
        [ 2.4855e-03,  2.2523e-03, -2.7343e-04],
        [-3.6921e-03, -3.3352e-03,  3.9381e-04],
        [-4.4827e-03, -4.0523e-03,  4.8165e-04],
        [ 1.6479e-03,  1.4775e-03, -1.6276e-04],
        [-3.9083e-04, -3.4578e-04,  3.3159e-05]], grad_fn=<DivBackward0>)

In [138]:
# Q = wq @ X
dwq = dQ.t() @ X
print(dwq)

tensor([[ 0.0010,  0.0015,  0.0022],
        [ 0.0009,  0.0014,  0.0019],
        [-0.0001, -0.0002, -0.0002]], grad_fn=<MmBackward0>)

In [140]:
# S = Q @ d.t() / sqrt(dim)
dK = dS.t() @ Q / math.sqrt(dim)
print(dK)

tensor([[ 1.0045e-05,  2.1119e-04, -6.4463e-04],
        [-1.6513e-05, -1.7858e-04,  5.6009e-04],
        [ 6.4244e-06,  3.0628e-05, -1.0284e-04],
        [ 2.0213e-05, -2.3253e-04,  6.5131e-04],
        [ 6.4723e-06,  4.6002e-05, -1.4849e-04],
        [-2.6642e-05,  1.2329e-04, -3.1544e-04]], grad_fn=<DivBackward0>)

In [142]:
# K = wk @ X
dwk = dK.t() @ X
print(dwk)

tensor([[-1.4332e-05, -6.1031e-06, -1.4288e-05],
        [-3.3169e-04,  4.3561e-04,  2.9329e-04],
        [ 1.0098e-03, -1.2795e-03, -8.4240e-04]], grad_fn=<MmBackward0>)

In [144]:
# Q, K, V = self.wq(X), self.wk(X), self.wv(X)
dXQ = dQ @ model.wq.weight
dXK = dK @ model.wk.weight
dXV = dV @ model.wv.weight
dX = dXQ + dXK + dXV
print(dX)

tensor([[ 0.0034, -0.0009, -0.0056],
        [ 0.0047, -0.0009, -0.0057],
        [ 0.0028, -0.0016, -0.0055],
        [ 0.0029, -0.0019, -0.0054],
        [ 0.0041, -0.0008, -0.0055],
        [ 0.0036, -0.0010, -0.0056]], grad_fn=<AddBackward0>)

In [146]:
# X = W X_Src
dw = dX.t() @ X_src
print(dw)

tensor([[ 0.0110,  0.0069,  0.0058],
        [-0.0028, -0.0042, -0.0031],
        [-0.0175, -0.0150, -0.0090]], grad_fn=<MmBackward0>)

### Forward

先实现带scale版本

In [20]:
def flash_attention(Q, K, V):
    O = torch.zeros_like(Q)
    
    for tq in range(block):     # q loop
        q = Q[tq*nb : (tq+1)*nb, :]
        o_old = torch.zeros_like(q)
    
        # global
        l_old = m_old = torch.zeros(nb, 1)
    
        # scale cache
        l_cache = torch.ones(nb, 1)
        
        for tk in range(block): # kv loop
            k = K[tk*nb : (tk+1)*nb , :]
            v = V[tk*nb : (tk+1)*nb , :]
    
            # score
            s = q @ k.t() / math.sqrt(dim)
    
            # local 
            m = torch.max(s, dim = 1, keepdim = True).values
            m_new = torch.maximum(m, m_old)
            l = torch.sum(torch.exp(s - m_new) , dim = 1, keepdim = True)

            # update global 
            l_new = l_old * torch.exp(m_old - m_new) + l
            l_cache = l_cache * l_new
    
            # update o
            o =  l_old * o_old * torch.exp(m_old - m_new) + torch.exp(s - m_new) @ v
            o = o / l_new
    
            # replace
            o_old = o
            l_old = l_new
            m_old = m_new

        O[tq*nb: (tq+1)*nb, :] = o
        # break

    return O

flash_attention(Q, K, V) 

tensor([[0.5620, 0.0820, 0.2353],
        [0.2852, 0.1104, 0.1981],
        [0.4802, 0.0893, 0.2232],
        [0.2864, 0.1118, 0.2006],
        [0.2819, 0.1196, 0.2113],
        [0.3466, 0.1088, 0.2142]], grad_fn=<CopySlices>)

外循环scale版本

In [21]:
def flash_attention(Q, K, V):
    O = torch.zeros_like(Q)
    
    for tq in range(block):     # q loop
        q = Q[tq*nb : (tq+1)*nb, :]
        o_old = torch.zeros_like(q)
    
        # global
        l_old = m_old = torch.zeros(nb, 1)
    
        # scale cache
        l_cache = torch.ones(nb, 1)
        
        for tk in range(block): # kv loop
            k = K[tk*nb : (tk+1)*nb , :]
            v = V[tk*nb : (tk+1)*nb , :]
    
            # score
            s = q @ k.t() / math.sqrt(dim)
    
            # local 
            m = torch.max(s, dim = 1, keepdim = True).values
            m_new = torch.maximum(m, m_old)
            l = torch.sum(torch.exp(s - m_new) , dim = 1, keepdim = True)

            # update global 
            l_new = l_old * torch.exp(m_old - m_new) + l
            l_cache = l_cache * l_new
    
            # update o
            o =  o_old * torch.exp(m_old - m_new) + torch.exp(s - m_new) @ v
            # o = o / l_new
    
            # replace
            o_old = o
            l_old = l_new
            m_old = m_new

        o = o_old / l_old
        O[tq*nb: (tq+1)*nb, :] = o

    return O

O = flash_attention(Q, K, V) 
print(O)

tensor([[0.5620, 0.0820, 0.2353],
        [0.2852, 0.1104, 0.1981],
        [0.4802, 0.0893, 0.2232],
        [0.2864, 0.1118, 0.2006],
        [0.2819, 0.1196, 0.2113],
        [0.3466, 0.1088, 0.2142]], grad_fn=<CopySlices>)

In [22]:
Y_pred = model.wo(O)
print(Y_pred)

tensor([[-0.1400, -0.3066,  0.3751],
        [-0.1367, -0.2193,  0.2119],
        [-0.1380, -0.2798,  0.3266],
        [-0.1385, -0.2213,  0.2132],
        [-0.1469, -0.2277,  0.2133],
        [-0.1431, -0.2439,  0.2500]], grad_fn=<MmBackward0>)

# Backward

分析实现

如果Q固定， 分析

前向

o = o_old / l_old

所以先计算

do_old = do / l_old

那么

o_new = o_old l_old e(m_old - m_new) - e(s - m_new) V